In [59]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

In [60]:
mat = Material("body")
rem = Material("P")
U = Field([mat, rem], type=:VectorField, dim=3, fieldName=:u)
Φ = Field([mat, rem], type=:VectorField, dim=3, fieldName=:φ);

In [61]:
cs = CoordinateSystem([-1, 0, 0], [0, -1, 0])
Q = rotateNodes(U, "left", cs)
cs1 = BoundaryCondition("left", ux=0, uy=0, uz=0)
dofs = constrainedDoFs(U, [cs1])
for i in dofs
    Q.T[i, i] = -1
end

In [62]:
Ku = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="body")
Ku = Q' * Ku * Q;

In [63]:
mpc_u = MPC(master="P", slave="right", field=U)

mpc_φ = MPC(master="P", slave="right", field=Φ);

In [64]:
R = rigidRotationMap(mpc_u, mpc_φ);

In [65]:
Kuφ = Ku * R;

In [66]:
Kφ = R' * Ku * R;

In [67]:
K = SystemMatrix([Ku Kuφ; Kuφ' Kφ]);

In [68]:
bc = BoundaryCondition("left", field=U, ux=0, uy=0, uz=0)

fu = ∫(U ⋅ [0, 0, 0], Γ="P")
fφ = ∫(Φ ⋅ [0, 0, 1e7], Γ="P")

F = SystemVector([fu, fφ]);

In [69]:
per = MPC(master="right", slave="left", field=U);

In [70]:
bcPu = BoundaryCondition("P", field=U, ux=0, uy=0, uz=0)

BoundaryCondition("P", Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38157, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))

In [71]:
u0, φ = solveField(K, F, support=[bcPu], mpc=[mpc_φ, per]);

In [72]:
u = u0 + R * φ;

In [73]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [74]:
u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u1 ∘ ∇ + ∇ ∘ u1) / 2

E = mat.E
ν = mat.μ
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [75]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")

n2 = normalVector(U, "mid")

σ2 = elementsToElements(σ, onPhysicalGroup="mid")

τ2 = σ2 * n2;

In [76]:
showElementResults(τ2, name="shear stress");

In [77]:
showDoFResults(n2, name="normal vector");

In [78]:
openPostProcessor()